# 🧠 Comprehensive Model Evaluation

This notebook evaluates all trained classification models on the test dataset and visualizes training curves.

**Models Evaluated:**
- ConvNeXt Base
- ResNet152V2
- DenseNet201
- EfficientNetV2S
- VGG16

**Outputs:**
1. Training curves for all models
2. Test set performance metrics (Accuracy, Precision, Recall, F1)
3. Confusion matrices
4. Updated README with results

## 1. Setup

In [9]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tqdm.notebook import tqdm

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import print as rprint

from src.classification.data import build_dataframe, make_dataset
from src.classification.inference import ClassificationEvaluator

# Initialize Rich Console
console = Console()

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

console.print(f"[bold green]TensorFlow version:[/bold green] {tf.__version__}")
console.print(f"[bold green]Project root:[/bold green] {PROJECT_ROOT}")

TensorFlow version: 2.20.0
Project root: d:\Learning\My projects\AI-Assisted-Brain-Tumor-Localization-and-Classification-in-Medical-Imaging


## 2. Load Training Logs

In [ ]:
# Define models and their log files
models_info = {
    'ConvNeXt': {
        'weights': 'weights/classification/ConvNeXtBase_best_weights.keras',
        'log': 'logs/classification/ConvNeXtBase_training_log.csv',
        'model_name': 'ConvNeXtBase'
    },
    'ResNet152V2': {
        'weights': 'weights/classification/ResNet152V2_best_weights.keras',
        'log': 'logs/classification/ResNet152V2_training_log.csv',
        'model_name': 'ResNet152V2'
    },
    'DenseNet201': {
        'weights': 'weights/classification/DenseNet201_best_weights.keras',
        'log': 'logs/classification/DenseNet201_training_log.csv',
        'model_name': 'DenseNet201'
    },
    'EfficientNetV2S': {
        'weights': 'weights/classification/EfficientNetV2S_best_weights.keras',
        'log': 'logs/classification/EfficientNetV2S_training_log.csv',
        'model_name': 'EfficientNetV2S'
    },
    'VGG16': {
        'weights': 'weights/classification/VGG16_best_weights.keras',
        'log': 'logs/classification/VGG16_training_log.csv',
        'model_name': 'VGG16'
    }
}

# Load training logs with progress bar
console.print("\n[bold cyan]📂 Loading Training Logs[/bold cyan]")
training_logs = {}

for name, info in tqdm(models_info.items(), desc="Loading logs", unit="model"):
    log_path = PROJECT_ROOT / info['log']
    if log_path.exists():
        training_logs[name] = pd.read_csv(log_path)
        console.print(f"  [green]✓[/green] {name}: [yellow]{len(training_logs[name])}[/yellow] epochs")
    else:
        console.print(f"  [red]✗[/red] Log file not found: {log_path}")

console.print(f"\n[bold green]Total models loaded:[/bold green] [yellow]{len(training_logs)}[/yellow]")

## 3. Training Curves Visualization

### 3.1 Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, log) in enumerate(training_logs.items()):
    ax = axes[idx]
    ax.plot(log['epoch'], log['loss'], label='Training Loss', linewidth=2)
    ax.plot(log['epoch'], log['val_loss'], label='Validation Loss', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Loss', fontsize=11)
    ax.set_title(f'{name} - Loss Curves', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Hide the last subplot if there are fewer than 6 models
if len(training_logs) < 6:
    axes[-1].axis('off')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'logs/classification/loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Accuracy Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, log) in enumerate(training_logs.items()):
    ax = axes[idx]
    ax.plot(log['epoch'], log['accuracy'] * 100, label='Training Accuracy', linewidth=2)
    ax.plot(log['epoch'], log['val_accuracy'] * 100, label='Validation Accuracy', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Accuracy (%)', fontsize=11)
    ax.set_title(f'{name} - Accuracy Curves', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

if len(training_logs) < 6:
    axes[-1].axis('off')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'logs/classification/accuracy_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Recall Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, log) in enumerate(training_logs.items()):
    ax = axes[idx]
    ax.plot(log['epoch'], log['overall_recall'] * 100, label='Training Recall', linewidth=2)
    ax.plot(log['epoch'], log['val_overall_recall'] * 100, label='Validation Recall', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Recall (%)', fontsize=11)
    ax.set_title(f'{name} - Recall Curves', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

if len(training_logs) < 6:
    axes[-1].axis('off')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'logs/classification/recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 AUC Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (name, log) in enumerate(training_logs.items()):
    ax = axes[idx]
    ax.plot(log['epoch'], log['auc_ovr'] * 100, label='Training AUC', linewidth=2)
    ax.plot(log['epoch'], log['val_auc_ovr'] * 100, label='Validation AUC', linewidth=2)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('AUC (%)', fontsize=11)
    ax.set_title(f'{name} - AUC Curves', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

if len(training_logs) < 6:
    axes[-1].axis('off')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'logs/classification/auc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Comparison - All Models

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Validation Loss
for name, log in training_logs.items():
    axes[0, 0].plot(log['epoch'], log['val_loss'], label=name, linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Validation Loss', fontsize=12)
axes[0, 0].set_title('Validation Loss Comparison', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Validation Accuracy
for name, log in training_logs.items():
    axes[0, 1].plot(log['epoch'], log['val_accuracy'] * 100, label=name, linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Validation Accuracy (%)', fontsize=12)
axes[0, 1].set_title('Validation Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Validation Recall
for name, log in training_logs.items():
    axes[1, 0].plot(log['epoch'], log['val_overall_recall'] * 100, label=name, linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=12)
axes[1, 0].set_ylabel('Validation Recall (%)', fontsize=12)
axes[1, 0].set_title('Validation Recall Comparison', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Validation AUC
for name, log in training_logs.items():
    axes[1, 1].plot(log['epoch'], log['val_auc_ovr'] * 100, label=name, linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=12)
axes[1, 1].set_ylabel('Validation AUC (%)', fontsize=12)
axes[1, 1].set_title('Validation AUC Comparison', fontsize=14, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'logs/classification/all_models_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Load Test Dataset

In [ ]:
console.print("\n[bold cyan]📊 Loading Test Dataset[/bold cyan]")
test_df = build_dataframe(split="test")
console.print(f"[green]Total test samples:[/green] [yellow]{len(test_df)}[/yellow]")

# Create table for class distribution
table = Table(title="Class Distribution", show_header=True, header_style="bold magenta")
table.add_column("Class", style="cyan")
table.add_column("Count", justify="right", style="yellow")

for class_name, count in test_df['class'].value_counts().items():
    table.add_row(class_name, str(count))

console.print(table)

## 5. Evaluate Models on Test Set

In [ ]:
results_data = []

# Evaluate models with progress bar
for name, info in tqdm(models_info.items(), desc="Evaluating models", unit="model"):
    console.print(Panel(f"[bold cyan]🔍 Evaluating {name}[/bold cyan]", expand=False))
    
    # Load model
    weights_path = PROJECT_ROOT / info['weights']
    if not weights_path.exists():
        console.print(f"[red]✗ Weights not found:[/red] {weights_path}")
        continue
    
    model = tf.keras.models.load_model(weights_path)
    console.print(f"[green]✓ Model loaded:[/green] {weights_path.name}")
    
    # Prepare dataset
    test_ds = make_dataset(
        test_df,
        model_name=info['model_name'],
        shuffle=False,
        batch_size=32,
    )
    
    # Create evaluator and evaluate
    evaluator = ClassificationEvaluator(model, test_ds)
    metrics = evaluator.evaluate(verbose=True)
    
    # Get per-class metrics
    per_class = evaluator.get_per_class_metrics()
    
    # Store results
    results_data.append({
        'Model': name,
        'Accuracy': metrics['sklearn_accuracy'],
        'Precision': metrics['macro_precision'],
        'Recall': metrics['macro_recall'],
        'F1': metrics['macro_f1'],
        'AUC': metrics.get('auc_ovr', 0.0)
    })
    
    # Print classification report
    evaluator.print_classification_report()
    
    # Plot confusion matrix
    cm_path = PROJECT_ROOT / f'logs/classification/{name}_confusion_matrix.png'
    evaluator.plot_confusion_matrix(normalize=True, save_path=str(cm_path))
    console.print(f"[green]💾 Confusion matrix saved:[/green] {cm_path.name}\n")

console.print("[bold green]✓ All models evaluated![/bold green]")

## 6. Results Summary

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame(results_data)
results_df = results_df.sort_values('Accuracy', ascending=False)

# Create Rich table
table = Table(title="📊 Test Set Results Summary", show_header=True, header_style="bold magenta")
table.add_column("Model", style="cyan", no_wrap=True)
table.add_column("Accuracy", justify="right", style="green")
table.add_column("Precision", justify="right", style="yellow")
table.add_column("Recall", justify="right", style="blue")
table.add_column("F1", justify="right", style="magenta")
table.add_column("AUC", justify="right", style="red")

for _, row in results_df.iterrows():
    table.add_row(
        row['Model'],
        f"{row['Accuracy']*100:.2f}%",
        f"{row['Precision']*100:.2f}%",
        f"{row['Recall']*100:.2f}%",
        f"{row['F1']*100:.2f}%",
        f"{row['AUC']*100:.2f}%"
    )

console.print(table)

# Display best model
best_model = results_df.iloc[0]['Model']
best_acc = results_df.iloc[0]['Accuracy']*100
console.print(f"\n[bold green]🏆 Best Performing Model:[/bold green] [bold cyan]{best_model}[/bold cyan] with [bold yellow]{best_acc:.2f}%[/bold yellow] accuracy")

# Display as styled table
results_df.style.highlight_max(axis=0, props='font-weight:bold;color:green')

### 6.1 Metrics Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar plot comparison
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1']
x = np.arange(len(results_df))
width = 0.2

for idx, metric in enumerate(metrics_to_plot):
    axes[0].bar(x + idx*width, results_df[metric]*100, width, label=metric)

axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('Score (%)', fontsize=12)
axes[0].set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(results_df['Model'])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')
axes[0].set_ylim([90, 100])

# Radar chart for best model
from math import pi

best_model_idx = 0
categories = metrics_to_plot
values = [results_df.iloc[best_model_idx][m]*100 for m in metrics_to_plot]
values += values[:1]  # Complete the circle

angles = [n / float(len(categories)) * 2 * pi for n in range(len(categories))]
angles += angles[:1]

ax = plt.subplot(1, 2, 2, projection='polar')
ax.plot(angles, values, 'o-', linewidth=2, label=results_df.iloc[best_model_idx]['Model'])
ax.fill(angles, values, alpha=0.25)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
ax.set_ylim(90, 100)
ax.set_title(f'Best Model: {results_df.iloc[best_model_idx]["Model"]}', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'logs/classification/test_results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Update README

In [ ]:
# Read current README
readme_path = PROJECT_ROOT / 'README.md'
with open(readme_path, 'r', encoding='utf-8') as f:
    readme_content = f.read()

# Create results table in markdown
results_table = "\n## 📈 Results\n\n"
results_table += "Test set performance on 1,000 T1-weighted MRI slices from the BRISC2025 dataset:\n\n"
results_table += "| Model | Accuracy | Precision | Recall | F1 Score | AUC |\n"
results_table += "|-------|----------|-----------|--------|----------|-----|\n"

for _, row in results_df.iterrows():
    results_table += f"| **{row['Model']}** | "
    results_table += f"{row['Accuracy']*100:.2f}% | "
    results_table += f"{row['Precision']*100:.2f}% | "
    results_table += f"{row['Recall']*100:.2f}% | "
    results_table += f"{row['F1']*100:.2f}% | "
    results_table += f"{row['AUC']*100:.2f}% |\n"

best_model = results_df.iloc[0]['Model']
best_acc = results_df.iloc[0]['Accuracy']*100
results_table += f"\n**Best Performing Model:** {best_model} with {best_acc:.2f}% accuracy\n\n"
results_table += "*All models use transfer learning with frozen backbones and custom classification heads.*\n"

# Find and replace the results section
import re
pattern = r'## 📈 Results.*?(?=##|$)'
updated_readme = re.sub(pattern, results_table, readme_content, flags=re.DOTALL)

# Write updated README
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(updated_readme)

console.print(Panel("[bold green]✓ README.md updated with test results![/bold green]", expand=False))
console.print("\n[bold cyan]Preview of results section:[/bold cyan]")
console.print(results_table)

## Summary

✅ **Completed Tasks:**
1. Loaded training logs for all 5 models
2. Visualized training curves (loss, accuracy, recall, AUC)
3. Evaluated all models on test dataset
4. Generated confusion matrices
5. Created performance comparison visualizations
6. Updated README.md with results table

**Generated Artifacts:**
- Training curve plots in `logs/classification/`
- Confusion matrices for each model
- Test results comparison chart
- Updated README.md